# GeoPandas: Mapping & Spatial Data

This notebook introduces **GeoPandas** — a Python library that extends pandas with spatial data types and operations — using a real **UK regions** shapefile.

## Contents
1. Reading a shapefile — UK regions
2. Exploring the GeoDataFrame
3. Choropleth map (colour by value)
4. Styled map with TG colours
5. Layered map — regions + cities
6. **Your Turn** — build your own analysis

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import numpy as np
from shapely.geometry import Point

import sys
sys.path.insert(0, '..')
from template import setup_matplotlib_template, get_colors, get_cmap

# Apply TG house style
setup_matplotlib_template('TG')
TG_COLORS = get_colors('TG')
tg_cmap = get_cmap('TG')

---
## 1. Reading a Shapefile — UK Regions

A **shapefile** is a common geospatial file format. It consists of several files (`.shp`, `.shx`, `.dbf`, `.prj`) that together define both the geometry and the attribute data.

GeoPandas reads them with `gpd.read_file()`.

In [ ]:
# Read the UK regions shapefile
gdf_regions = gpd.read_file('../assets/UK_counties/georef-united-kingdom-region-millesime.shp')

gdf_regions

---
## 2. Exploring the GeoDataFrame

Let's clean up the data and understand what we're working with.

In [ ]:
# The column values have square brackets — clean them up
for col in ['ctry_code', 'ctry_name', 'rgn_code', 'rgn_name']:
    gdf_regions[col] = gdf_regions[col].str.strip("[]'")

In [ ]:
# Quick plot
fig, ax = plt.subplots(figsize=(3, 4))
gdf_regions.plot(ax=ax, color=TG_COLORS[0], edgecolor='k', linewidth=0.5)
ax.set_title('UK Regions')
ax.axis('off')
plt.tight_layout()
plt.show()

---
## 3. Choropleth Map (Colour by Value)

A **choropleth** colours each region according to a numeric value. We'll create some made-up data and merge it onto the GeoDataFrame.

In [ ]:
# Made-up regional statistics
region_stats = pd.DataFrame({
    'rgn_name': ['London', 'South East', 'South West', 'East of England',
                 'West Midlands', 'East Midlands', 'North West',
                 'Yorkshire and The Humber', 'North East',
                 'Scotland', 'Wales', 'Northern Ireland'],
    'projects':     [152, 87, 64, 71, 58, 43, 76, 51, 32, 68, 37, 24],
    'funding_m':    [385, 195, 142, 168, 130, 98, 178, 115, 72, 155, 85, 54],
    'universities': [40,  18,  12,  10,  13,   9,  16,  11,   5,  15,   8,   4],
})
region_stats

In [ ]:
# Merge onto the GeoDataFrame
gdf = gdf_regions.merge(region_stats, on='rgn_name', how='right')
gdf

In [ ]:
# Choropleth — colour by total funding
fig, ax = plt.subplots(figsize=(3, 4))
gdf.plot(ax=ax, column='funding_m', cmap=tg_cmap, edgecolor='k',
         linewidth=0.5, legend=True,
         legend_kwds={'label': 'Funding (£m)', 'shrink': 0.4})

ax.set_title('Research Funding by UK Region')
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Flat colour map with funding amounts annotated on each region
fig, ax = plt.subplots(figsize=(3, 4))
gdf.plot(ax=ax, column='funding_m', cmap=tg_cmap, edgecolor='k', linewidth=0.5)

for _, row in gdf.iterrows():
    centroid = row.geometry.centroid
    ax.annotate(f"£{row['funding_m']:.0f}m",
                xy=(centroid.x, centroid.y),
                ha='center', va='center', fontsize=4, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor='none'))

ax.set_title('Research Funding by UK Region (£m)')
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Create label position columns from centroids — you can edit these to reposition labels
gdf['label_x'] = gdf.geometry.centroid.x
gdf['label_y'] = gdf.geometry.centroid.y

# Uncomment and tweak to nudge specific labels, e.g.:
gdf.loc[gdf['rgn_name'] == 'London', 'label_x'] += 1
gdf.loc[gdf['rgn_name'] == 'South East', 'label_y'] -= 0.2

In [ ]:
# Re-plot using the editable label positions
fig, ax = plt.subplots(figsize=(3, 4))
gdf.plot(ax=ax, column='funding_m', cmap=tg_cmap, edgecolor='k', linewidth=0.5)

for _, row in gdf.iterrows():
    ax.annotate(f"£{row['funding_m']:.0f}m",
                xy=(row['label_x'], row['label_y']),
                ha='center', va='center', fontsize=4, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.9, edgecolor='none'))

ax.set_title('Research Funding by UK Region (£m)')
ax.axis('off')
plt.tight_layout()
plt.show()

---
## Your Turn — CORDIS Funding by Country

Using what you've learned above, create a **choropleth map of Europe** showing EU research funding per country.

### Data
- **CORDIS organisations**: `../assets/cordis_orgs.csv` — one row per organisation per project, with columns `vatNumber`, `ecContribution`, `masterCall`, etc.
- **Europe shapefile**: `../assets/europe_countries/world-administrative-boundaries.shp` — 52 European countries with an `iso_3166_1_` column (2-letter country code like `FR`, `DE`)

### Steps

1. **Load the CORDIS data** and filter to a single master call (e.g. `'HORIZON-CL5-2022-D2-01'`)
2. **Extract the country code** from the first 2 letters of `vatNumber`:
   ```python
   df['country'] = df['vatNumber'].str.extract(r'^([A-Z]{2})')
   ```
3. **Aggregate funding per country**:
   ```python
   funding_by_country = df.groupby('country')['ecContribution'].sum().reset_index()
   funding_by_country.columns = ['iso_3166_1_', 'total_funding']
   ```
4. **Load the Europe shapefile** and merge the funding data onto it using `iso_3166_1_` as the join key
5. **Plot a choropleth** coloured by `total_funding`